In [4]:
import pyspark
print(pyspark.__version__)

2.4.1


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

spark = (
    SparkSession.builder
    .appName("RetailPulseBronzeIngestion")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0")
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/hive/warehouse")
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
    .getOrCreate()
)

In [6]:
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("event_timestamp", StringType()),
    StructField("customer_id", IntegerType()),
    StructField("session_id", StringType()),
    StructField("order_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("store_id", IntegerType()),
    StructField("channel", StringType()),
    StructField("sequence", IntegerType()),
])

In [7]:
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "retail_logs")
    .option("startingOffsets", "earliest")
    .load()
)

parsed_df = raw_df.selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_ingest_ts")

bronze_df = (
    parsed_df
    .select(from_json(col("json_str"), event_schema).alias("data"), col("json_str"), col("kafka_ingest_ts"))
    .select("data.*", col("json_str").alias("raw_payload"), "kafka_ingest_ts")
)

In [8]:
hdfs_bronze_path = "hdfs://namenode:8020/data/bronze/retailpulse_events"
checkpoint_path = "hdfs://namenode:8020/checkpoints/retailpulse_events_bronze"

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS bronze.retailpulse_events (
    event_id STRING,
    event_type STRING,
    event_timestamp STRING,
    customer_id INT,
    session_id STRING,
    order_id INT,
    product_id INT,
    store_id INT,
    channel STRING,
    sequence INT,
    raw_payload STRING,
    kafka_ingest_ts TIMESTAMP
)
PARTITIONED BY (ingestion_date DATE)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LINES TERMINATED BY '\\n'
STORED AS TEXTFILE
LOCATION '{hdfs_bronze_path}'
TBLPROPERTIES ("skip.header.line.count"="1")
""")

DataFrame[]

In [9]:
def write_to_bronze(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return
    enriched = batch_df.withColumn("ingestion_date", to_date(col("kafka_ingest_ts")))
    (
        enriched.write
        .mode("append")
        .partitionBy("ingestion_date")
        .option("header", "true")
        .csv(hdfs_bronze_path)
    )
    spark.sql("MSCK REPAIR TABLE bronze.retailpulse_events")

query = (
    bronze_df
    .writeStream
    .foreachBatch(write_to_bronze)
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(processingTime="30 seconds")
    .start()
)

query.awaitTermination()

KeyboardInterrupt: 